In [55]:
import duckdb
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo
import os

In [56]:
WAREHOUSE_TEST = "./test_duckdb_data/gtftest7.duckdb"

con = duckdb.connect(WAREHOUSE_TEST)
con.execute("SET TimeZone='UTC';")

con.sql("SHOW TABLES").df()

,name
0,agency
1,calendar
2,calendar_dates
3,feed_info
4,routes
5,shapes
6,stop_times
7,stops
8,trips
9,trips_updates


In [57]:
con.sql("DESCRIBE trips_updates").df()


,column_name,column_type,null,key,default,extra
0,trip_id,VARCHAR,YES,None,None,None
1,route_id,VARCHAR,YES,None,None,None
2,direction_id,DOUBLE,YES,None,None,None
3,stop_id,BIGINT,YES,None,None,None
4,stop_sequence,BIGINT,YES,None,None,None
5,arrival_time,DOUBLE,YES,None,None,None
6,departure_time,DOUBLE,YES,None,None,None
7,arrival_dt,TIMESTAMP_NS,YES,None,None,None
8,departure_dt,TIMESTAMP_NS,YES,None,None,None


In [58]:
con.sql("DESCRIBE vehicle_positions").df()


,column_name,column_type,null,key,default,extra
0,trip_id,VARCHAR,YES,None,None,None
1,route_id,VARCHAR,YES,None,None,None
2,stop_id,BIGINT,YES,None,None,None
3,latitude,DOUBLE,YES,None,None,None
4,longitude,DOUBLE,YES,None,None,None
5,bearing,DOUBLE,YES,None,None,None
6,speed,DOUBLE,YES,None,None,None
7,timestamp,BIGINT,YES,None,None,None
8,vehicle_id,VARCHAR,YES,None,None,None
9,timestamp_dt,TIMESTAMP_NS,YES,None,None,None


In [59]:
# donnees temmps reel
con.sql("""
CREATE OR REPLACE TEMP VIEW actual AS
SELECT
  tu.trip_id,
  tu.stop_sequence,
  CAST(tu.arrival_dt AS TIMESTAMP WITH TIME ZONE) AS arrival_dt_utc
FROM trips_updates tu
WHERE tu.arrival_dt IS NOT NULL;
""")

con.sql("SELECT * FROM actual LIMIT 10").df()

,trip_id,stop_sequence,arrival_dt_utc
0,6170769-L3_A_14__22:07-JANV2025-L3-Semaine-09,2,2025-09-10 20:09:20+00:00
1,6170769-L3_A_14__22:07-JANV2025-L3-Semaine-09,3,2025-09-10 20:11:27+00:00
2,6170769-L3_A_14__22:07-JANV2025-L3-Semaine-09,4,2025-09-10 20:12:27+00:00
3,6170769-L3_A_14__22:07-JANV2025-L3-Semaine-09,5,2025-09-10 20:14:27+00:00
4,6170769-L3_A_14__22:07-JANV2025-L3-Semaine-09,6,2025-09-10 20:16:27+00:00
5,6471097-22_A_45_2213_22:30-PROJET2025-22-Mercr...,3,2025-09-10 20:36:51+00:00
6,6471097-22_A_45_2213_22:30-PROJET2025-22-Mercr...,6,2025-09-10 20:56:00+00:00
7,6471097-22_A_45_2213_22:30-PROJET2025-22-Mercr...,7,2025-09-10 20:57:09+00:00
8,6471097-22_A_45_2213_22:30-PROJET2025-22-Mercr...,8,2025-09-10 20:58:39+00:00
9,6471097-22_A_45_2213_22:30-PROJET2025-22-Mercr...,9,2025-09-10 20:59:21+00:00


In [60]:
# time theorique

con.sql("""
CREATE OR REPLACE TEMP VIEW sched AS
SELECT
  st.trip_id,
  CAST(st.stop_sequence AS BIGINT) AS stop_sequence,
  st.stop_id,
  st.arrival_time_sec
FROM stop_times st
WHERE st.arrival_time_sec IS NOT NULL;
""")

con.sql("SELECT * FROM sched LIMIT 10").df()


,trip_id,stop_sequence,stop_id,arrival_time_sec
0,3064029-C32_A_1_C3201_08:30-RESEAU2021-C32-Lun...,0,21681,30600
1,3064029-C32_A_1_C3201_08:30-RESEAU2021-C32-Lun...,1,21652,30900
2,3064029-C32_A_1_C3201_08:30-RESEAU2021-C32-Lun...,2,21644,32100
3,3064029-C32_A_1_C3201_08:30-RESEAU2021-C32-Lun...,3,21398,33300
4,3064030-C32_R_2_C3202_17:20-RESEAU2021-C32-Lun...,0,21398,62400
5,3064030-C32_R_2_C3202_17:20-RESEAU2021-C32-Lun...,1,21644,63600
6,3064030-C32_R_2_C3202_17:20-RESEAU2021-C32-Lun...,2,21652,64800
7,3064030-C32_R_2_C3202_17:20-RESEAU2021-C32-Lun...,3,21681,65100
8,3064033-C32_A_1_C3202_17:00-RESEAU2021-C32-Wee...,0,21681,61200
9,3064033-C32_A_1_C3202_17:00-RESEAU2021-C32-Wee...,1,21652,61500


In [61]:
# join entre time theorique - donnees reeel

con.sql("""
CREATE OR REPLACE TEMP VIEW joined AS
SELECT
  a.trip_id,
  a.stop_sequence,
  s.stop_id,
  a.arrival_dt_utc,
  CAST(a.arrival_dt_utc AT TIME ZONE 'Europe/Paris' AS DATE) AS local_service_day,
  s.arrival_time_sec
FROM actual a
JOIN sched s
  ON a.trip_id = s.trip_id
 AND a.stop_sequence = s.stop_sequence;
""")

con.sql("SELECT * FROM joined LIMIT 10").df()


,trip_id,stop_sequence,stop_id,arrival_dt_utc,local_service_day,arrival_time_sec
0,4601755-44_R_17_4401_23:08-RESEAU2023-44-Semai...,1,4242,2025-09-10 21:09:03+00:00,2025-09-10,83340
1,4601755-44_R_17_4401_23:08-RESEAU2023-44-Semai...,3,4243,2025-09-10 21:10:28+00:00,2025-09-10,83400
2,4601755-44_R_17_4401_23:08-RESEAU2023-44-Semai...,4,4266,2025-09-10 21:11:20+00:00,2025-09-10,83460
3,4601755-44_R_17_4401_23:08-RESEAU2023-44-Semai...,5,4244,2025-09-10 21:12:42+00:00,2025-09-10,83520
4,4601755-44_R_17_4401_23:08-RESEAU2023-44-Semai...,6,4245,2025-09-10 21:14:38+00:00,2025-09-10,83640
5,4601756-44_A_13_4401_23:00-RESEAU2023-44-Semai...,2,4297,2025-09-10 21:02:28+00:00,2025-09-10,82920
6,4601756-44_A_13_4401_23:00-RESEAU2023-44-Semai...,3,4239,2025-09-10 21:03:04+00:00,2025-09-10,82980
7,4601756-44_A_13_4401_23:00-RESEAU2023-44-Semai...,4,4240,2025-09-10 21:05:10+00:00,2025-09-10,83100
8,4601756-44_A_13_4401_23:00-RESEAU2023-44-Semai...,6,4241,2025-09-10 21:07:19+00:00,2025-09-10,83220
9,4601756-44_A_13_4401_23:00-RESEAU2023-44-Semai...,7,4135,2025-09-10 21:08:00+00:00,2025-09-10,83280


In [62]:
# retard 

con.sql("""
CREATE OR REPLACE TEMP VIEW per_event AS
SELECT
  trip_id,
  stop_sequence,
  stop_id,
  arrival_dt_utc,
  (
    (local_service_day + arrival_time_sec * INTERVAL '1 second')
    AT TIME ZONE 'Europe/Paris'
  ) AS scheduled_ts_utc,
  EXTRACT(
    EPOCH FROM (
      arrival_dt_utc - (
        (local_service_day + arrival_time_sec * INTERVAL '1 second')
        AT TIME ZONE 'Europe/Paris'
      )
    )
  ) / 60.0 AS delay_min
FROM joined;
""")

con.sql("SELECT * FROM per_event ORDER BY arrival_dt_utc DESC LIMIT 10").df()


,trip_id,stop_sequence,stop_id,arrival_dt_utc,scheduled_ts_utc,delay_min
0,6429342-15_R_99_1502_23:40-SETP2025-15-Semaine-11,28,463,2025-09-10 22:08:00+00:00,2025-09-11 22:08:00+00:00,-1440.000000
1,6409672-57_R_99_5703_24:05-SETP2025-57-Semaine-06,4,525,2025-09-10 22:07:11+00:00,2025-09-10 22:07:00+00:00,0.183333
2,6409682-57_A_50_5707_24:05-SETP2025-57-Semaine-06,3,703,2025-09-10 22:07:00+00:00,2025-09-10 22:07:00+00:00,0.000000
3,6409671-57_R_99_5712_23:35-SETP2025-57-Semaine-06,33,1150,2025-09-10 22:07:00+00:00,2025-09-11 22:07:00+00:00,-1440.000000
4,6409682-57_A_50_5707_24:05-SETP2025-57-Semaine-06,2,87,2025-09-10 22:06:13+00:00,2025-09-10 22:06:00+00:00,0.216667
5,6409671-57_R_99_5712_23:35-SETP2025-57-Semaine-06,32,313,2025-09-10 22:05:43+00:00,2025-09-11 22:06:00+00:00,-1440.283333
6,6409681-57_A_50_5709_23:35-SETP2025-57-Semaine-06,28,279,2025-09-10 22:05:39+00:00,2025-09-11 22:06:00+00:00,-1440.350000
7,6409672-57_R_99_5703_24:05-SETP2025-57-Semaine-06,1,773,2025-09-10 22:05:34+00:00,2025-09-10 22:06:00+00:00,-0.433333
8,6409682-57_A_50_5707_24:05-SETP2025-57-Semaine-06,1,314,2025-09-10 22:05:33+00:00,2025-09-10 22:06:00+00:00,-0.450000
9,6429342-15_R_99_1502_23:40-SETP2025-15-Semaine-11,26,179,2025-09-10 22:05:24+00:00,2025-09-11 22:05:00+00:00,-1439.600000


In [63]:
# positions vehicles

con.sql("""
CREATE OR REPLACE TEMP VIEW vp AS
SELECT
  trip_id,
  route_id,
  latitude,
  longitude,
  to_timestamp("timestamp") AS ts_utc
FROM vehicle_positions;
""")

con.sql("SELECT * FROM vp ORDER BY ts_utc DESC LIMIT 10").df()



,trip_id,route_id,latitude,longitude,ts_utc
0,6402598-32_A_48_3202_22:15-SETP2025-32-Semaine-34,32,43.676628,7.228001,2025-09-10 20:09:43+00:00
1,5203704-68_A_50_6806_21:53-PROJET2024-68-Semai...,68,43.800003,7.193683,2025-09-10 20:09:43+00:00
2,5203736-68_R_99_6805_22:00-PROJET2024-68-Semai...,68,43.796535,7.202498,2025-09-10 20:09:43+00:00
3,6409685-57_A_50_5709_21:40-SETP2025-57-Semaine-06,57,43.715740,7.253726,2025-09-10 20:09:43+00:00
4,6444239-17_R_99_1704_22:10-SETP2025-17-Mercred...,17,43.691414,7.194284,2025-09-10 20:09:43+00:00
5,6471096-22_R_95_2213_21:50-PROJET2025-22-Mercr...,22,43.690804,7.195817,2025-09-10 20:09:43+00:00
6,6429343-15_A_50_1504_21:50-SETP2025-15-Semaine-11,15,43.701839,7.325854,2025-09-10 20:09:43+00:00
7,6369764-42_R_95_4203_18:50-PROJET2025-42-Semai...,42,43.685448,7.149457,2025-09-10 20:09:43+00:00
8,6368769-09_R_99_0907_22:30-PROJET2025-09-Semai...,09,43.669331,7.210083,2025-09-10 20:09:43+00:00
9,6471085-22_A_45_2210_22:00-PROJET2025-22-Mercr...,22,43.707947,7.184255,2025-09-10 20:09:43+00:00


In [64]:
# retard matched

con.sql("""
CREATE OR REPLACE TEMP VIEW matched AS
SELECT
  v.trip_id,
  v.route_id,
  v.latitude,
  v.longitude,
  v.ts_utc,
  pe.delay_min,
  pe.arrival_dt_utc AS matched_event_ts
FROM vp v
LEFT JOIN LATERAL (
  SELECT arrival_dt_utc, delay_min
  FROM per_event
  WHERE trip_id = v.trip_id
    AND arrival_dt_utc <= v.ts_utc
  ORDER BY arrival_dt_utc DESC
  LIMIT 1
) pe ON TRUE;
""")

con.sql("SELECT * FROM matched ORDER BY ts_utc DESC LIMIT 10").df()


,trip_id,route_id,latitude,longitude,ts_utc,delay_min,matched_event_ts
0,6257264-63_A_48_6305_22:00-PROJET2025-63-Mercr...,63,43.713955,7.250575,2025-09-10 20:09:43+00:00,-0.050000,2025-09-10 20:08:57+00:00
1,6409678-57_A_50_5707_22:05-SETP2025-57-Semaine-06,57,43.699745,7.286570,2025-09-10 20:09:43+00:00,1.216667,2025-09-10 20:09:13+00:00
2,6429339-15_R_99_1502_22:00-SETP2025-15-Semaine-11,15,43.704224,7.321824,2025-09-10 20:09:43+00:00,-2.450000,2025-09-10 20:09:33+00:00
3,6429343-15_A_50_1504_21:50-SETP2025-15-Semaine-11,15,43.701839,7.325854,2025-09-10 20:09:43+00:00,-0.800000,2025-09-10 20:09:12+00:00
4,6443781-07_A_49_0705_21:47-SETP2025-07-Mercred...,07,43.708286,7.287794,2025-09-10 20:09:43+00:00,1.200000,2025-09-10 20:08:12+00:00
5,6447023-05_A_99_0510_22:00-SETP2025-05-Mercred...,05,43.708843,7.270878,2025-09-10 20:09:43+00:00,1.216667,2025-09-10 20:09:13+00:00
6,6409668-57_R_99_5703_22:05-SETP2025-57-Semaine-06,57,43.725609,7.255367,2025-09-10 20:09:43+00:00,-2.116667,2025-09-10 20:08:53+00:00
7,6428496-61_R_99_6102_22:00-PROJET2025-61-Semai...,61,43.706238,7.220822,2025-09-10 20:09:43+00:00,2.400000,2025-09-10 20:09:24+00:00
8,5203704-68_A_50_6806_21:53-PROJET2024-68-Semai...,68,43.800003,7.193683,2025-09-10 20:09:43+00:00,-0.350000,2025-09-10 20:09:39+00:00
9,6428334-60_R_99_6002_22:00-PROJET2025-60-Semai...,chouette:Line:4a8dc505-72e7-4b72-b186-a5b46e94...,43.697903,7.237248,2025-09-10 20:09:43+00:00,-2.483333,2025-09-10 20:09:31+00:00


In [65]:
# couverture
con.sql("""
SELECT
  COUNT(*) AS n_positions,
  SUM(CASE WHEN delay_min IS NOT NULL THEN 1 ELSE 0 END) AS n_matched,
  ROUND(100.0 * SUM(CASE WHEN delay_min IS NOT NULL THEN 1 ELSE 0 END)/COUNT(*), 2) AS coverage_pct
FROM matched;
""").df()

,n_positions,n_matched,coverage_pct
0,61,19.0,31.15


In [66]:
EXPORT_DIR = "./exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

def current_timestamp_string():
    return datetime.now(ZoneInfo("Europe/Paris")).strftime("%Y%m%d_%H%M%S")

df = con.sql("""
SELECT
  trip_id,
  route_id,
  latitude,
  longitude,
  ts_utc AT TIME ZONE 'Europe/Paris' AS timestamp_paris,
  ROUND(delay_min, 2) AS delay_min
FROM matched
ORDER BY ts_utc DESC;
""").df()

csv_path = f"{EXPORT_DIR}/position_bus_vehicles_{current_timestamp_string()}.csv"
df.to_csv(csv_path, index=False)
csv_path

'./exports/position_bus_vehicles_20250910_235109.csv'